# Blackjack — Reinforcement Learning Workshop
### Based on: *Beating Blackjack — A Reinforcement Learning Approach* (Geiser & Hasseler, Stanford)

This notebook walks through a Blackjack simulation used to train and evaluate Q-Learning (and related) agents.

**Run order:**
1. Imports & parameters
2. The game environment
3. Generate training data (random policy)
4. Evaluate a trained policy

---

## Cell 1 — Imports

We only need standard scientific Python libraries.

In [1]:
import os
import numpy as np
import pandas as pd
import random

## Cell 2 — Parameters

All settings live here. Change `action_type` to switch modes:
- `'random_policy'` — plays randomly, saves training data to CSV
- `'fixed_policy'` — loads a `.policy` file and evaluates it
- `'input'` — lets you play manually in the terminal

Change `state_mapping` between `1` and `2` to toggle how the game state is represented (more on this below).

In [2]:
class Params():
    def __init__(self):
        # 'input', 'random_policy', or 'fixed_policy'
        self.action_type = 'random_policy'

        # Number of games to simulate
        self.num_games = 100000

        # Path to .policy file (only used when action_type = 'fixed_policy')
        self.fixed_policy_filepath = os.path.join(os.getcwd(), 'QLearning_policy_mapping_2.policy')

        # State mapping: 1 = player hand only, 2 = player hand + dealer upcard
        self.state_mapping = 2

## Cell 3 — State Mapping Explained

The *state* is how the agent perceives the world. There are two options:

**State mapping 1** — only the player's hand total (18 values: hands 4–21, plus win/lose/terminal)
```
state = player_hand_sum - 1
```
Total: **21 states**

**State mapping 2** — player's hand total *and* dealer's visible card
```
state = (player_hand_sum - 1) + (18 × (dealer_upcard - 1))
```
Total: **183 states** — more info, better decisions, but needs more training data.

States 0, 1, 2 are reserved for lose, win, and terminal.

## Cell 4 — The Game Environment

This class handles everything about a Blackjack game: the deck, drawing cards, determining who won, and mapping hands to states.

Key methods:
- `hand_to_state()` — converts the current game situation into a state index
- `get_reward()` — returns the reward signal for a given action
- `play_game()` — plays one full game and records (s, a, r, s') tuples
- `play_games()` — runs many games and optionally saves the data

In [3]:
class BlackJack_game():
    def __init__(self, params):
        # Standard deck: Ace=1, face cards=10
        self.deck = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10] * 4
        random.shuffle(self.deck)

        self.player = self.draw_hand()
        self.dealer = [self.draw_card()]

        # Stores (state, action, reward, next_state) tuples per game
        self.sarsp = []
        self.sarsp_arr = np.array([], dtype='int').reshape(0, 4)

        self.action_type = params.action_type
        self.verbose = (params.action_type == 'input')
        self.num_games = params.num_games
        self.fixed_policy_filepath = params.fixed_policy_filepath
        self.policy = self.load_policy()
        self.state_mapping = params.state_mapping

        # Reserved state indices
        self.lose_state = 0
        self.win_state = 1
        self.terminal_state = 2

        # Terminal rewards
        self.lose_reward = -10
        self.win_reward = 10

    # ── Deck helpers ──────────────────────────────────────────────────────────

    def reset(self):
        """Reset the deck and hands for a new game."""
        self.player = self.draw_hand()
        self.dealer = [self.draw_card()]
        self.sarsp = []
        self.deck = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10] * 4
        random.shuffle(self.deck)

    def draw_card(self):
        return self.deck.pop()

    def draw_hand(self):
        return [self.draw_card(), self.draw_card()]

    # ── Hand evaluation ───────────────────────────────────────────────────────

    def usable_ace(self, hand):
        """True if the hand has an Ace that can count as 11 without busting."""
        return 1 in hand and sum(hand) + 10 <= 21

    def sum_hand(self, hand):
        """Return the best total for a hand (counts Ace as 11 if possible)."""
        if self.usable_ace(hand):
            return sum(hand) + 10
        return sum(hand)

    def is_bust(self, hand):
        return self.sum_hand(hand) > 21

    def score(self, hand):
        return 0 if self.is_bust(hand) else self.sum_hand(hand)

    def player_won(self, player, dealer):
        """Return True if the player beat the dealer."""
        if self.is_bust(player):    return False
        elif self.is_bust(dealer):  return True
        elif self.sum_hand(player) > self.sum_hand(dealer): return True
        else: return False

    # ── State & reward ────────────────────────────────────────────────────────

    def hand_to_state(self, player, dealer):
        """Convert current hands to a single integer state index."""
        if self.state_mapping == 1:
            return self.sum_hand(player) - 1
        elif self.state_mapping == 2:
            return (self.sum_hand(player) - 1) + (18 * (dealer[0] - 1))

    def get_reward(self, state, action, player, dealer):
        """Small shaping rewards for obviously good/bad moves (mapping 2 only)."""
        if self.state_mapping == 1:
            return 0
        else:
            if (self.sum_hand(player) <= 11 and action == 1) or \
               (self.sum_hand(player) >= 17 and action == 0):
                return 1   # clearly good move
            elif (self.sum_hand(player) <= 11 and action == 0) or \
                 (self.sum_hand(player) >= 17 and action == 1):
                return -1  # clearly bad move
            else:
                return 0

    # ── Policy ────────────────────────────────────────────────────────────────

    def load_policy(self):
        """Load a saved .policy file (one action per line)."""
        if self.action_type in ['random_policy', 'input']:
            return None
        with open(self.fixed_policy_filepath, 'r') as f:
            return [int(x) for x in f.read().split()]

    def get_action(self, state):
        """Return action (0=stand, 1=hit) based on current mode."""
        if self.action_type == 'input':
            return int(input('Hit (1) or Stand (0): '))
        elif self.action_type == 'random_policy':
            return np.random.randint(2)
        elif self.action_type == 'fixed_policy':
            return self.policy[state]

    # ── Game loop ─────────────────────────────────────────────────────────────

    def play_game(self):
        """Play one full game, recording (s, a, r, s') at each step."""
        if self.verbose:
            print('New Game!\n')

        done = False
        while not done:
            self.print_iter()
            state  = self.hand_to_state(self.player, self.dealer)
            action = self.get_action(state)
            reward = self.get_reward(state, action, self.player, self.dealer)

            if action:  # hit
                self.player.append(self.draw_card())
                done = self.is_bust(self.player)
            else:       # stand — dealer plays out
                while self.sum_hand(self.dealer) < 17:
                    self.dealer.append(self.draw_card())
                done = True

            if not done:
                sp = self.hand_to_state(self.player, self.dealer)
                self.sarsp.append([state, action, reward, sp])

        self.print_iter()
        won = self.player_won(self.player, self.dealer)
        sp  = self.win_state if won else self.lose_state
        self.sarsp.append([state, action, reward, sp])

        # Final terminal transition
        reward = self.win_reward if won else self.lose_reward
        self.sarsp.append([sp, np.random.randint(2), reward, self.terminal_state])

        if self.verbose:
            print(f'Player won?: {won}')

        self.sarsp_arr = np.vstack((self.sarsp_arr, np.array(self.sarsp)))

    def print_iter(self):
        if not self.verbose:
            return
        print(f'Player hand: {self.player}\t sum: {self.sum_hand(self.player)}')
        print(f'Dealer hand: {self.dealer}\t sum: {self.sum_hand(self.dealer)}')

    # ── Multi-game runner ─────────────────────────────────────────────────────

    def output_sarsp_file(self):
        """Save collected (s, a, r, s') data to CSV for training."""
        filename = f'random_policy_runs_mapping_{self.state_mapping}.csv'
        output_filepath = os.path.join(os.getcwd(), filename)
        pd.DataFrame(self.sarsp_arr, columns=['s','a','r','sp']).to_csv(output_filepath, index=None)
        print(f'Saved training data to {output_filepath}')

    def print_stats(self):
        num_wins = np.count_nonzero(self.sarsp_arr[:, 0] == self.win_state)
        num_lose = np.count_nonzero(self.sarsp_arr[:, 0] == self.lose_state)
        print(f'Games played:   {self.num_games}')
        print(f'Wins:           {num_wins}')
        print(f'Losses:         {num_lose}')
        print(f'Win percentage: {num_wins / self.num_games:.1%}')

    def play_games(self):
        """Simulate num_games games and print stats. Saves CSV if random policy."""
        for i in range(self.num_games):
            self.play_game()
            self.reset()
        self.print_stats()
        if self.action_type == 'random_policy':
            self.output_sarsp_file()

## Cell 5 — Generate Training Data

Run this cell to simulate games using a **random policy**. The agent hits or stands randomly, and all `(state, action, reward, next_state)` tuples are saved to a CSV.

This CSV is the training input for Q-Learning, SARSA, and Value Iteration.

> For state mapping 2, use at least **100,000 games** to cover enough state-action pairs.

In [4]:
params = Params()
params.action_type = 'random_policy'
params.num_games = 100000
params.state_mapping = 2

game = BlackJack_game(params)
game.play_games()

Games played:   100000
Wins:           28022
Losses:         71978
Win percentage: 28.0%
Saved training data to c:\Users\matt\Documents\[1] - Code Projects\[0] - College\Autonomous-systems-Portfolio\Workshop\blackjack_project\notebooks\random_policy_runs_mapping_2.csv


## Cell 6 — Inspect the Training Data

Let's look at what was saved. Each row is one step in a game:
- `s` — current state
- `a` — action taken (0 = stand, 1 = hit)
- `r` — reward received
- `sp` — next state

In [5]:
df = pd.read_csv('random_policy_runs_mapping_2.csv')
print(f'Total rows: {len(df)}')
print(f'Unique states visited: {df["s"].nunique()} / 183')
df.head(10)

Total rows: 237396
Unique states visited: 182 / 183


,s,a,r,sp
0,179,1,-1,0
1,0,0,-10,2
2,52,0,1,0
3,0,0,-10,2
4,78,0,-1,1
5,1,1,10,2
6,164,1,-1,160
7,160,0,1,0
8,0,1,-10,2
9,128,1,-1,128


## Cell 7 — Evaluate a Trained Policy

Once you've trained a policy with `q_learning.py` (or SARSA / Value Iteration), load it here and run it against the dealer.

Change `fixed_policy_filepath` to point to whichever `.policy` file you want to evaluate.

In [8]:
params = Params()
params.action_type = 'fixed_policy'
params.num_games = 100000
params.state_mapping = 2
params.fixed_policy_filepath = os.path.join(os.getcwd(), 'QLearning_policy_mapping_2.policy')

game = BlackJack_game(params)
game.play_games()

Games played:   100000
Wins:           42015
Losses:         57985
Win percentage: 42.0%


## Cell 8 — Compare Policies

Run multiple policies and compare their win rates side by side.

In [11]:
policy_files = [
    ('Q-Learning',       'QLearning_policy_mapping_2.policy'),
    ('SARSA',            'Sarsa_Policy_2.policy'),
    ('Value Iteration',  'Value_Iteration_Policy_2.policy'),
]

results = []
for name, filepath in policy_files:
    if not os.path.exists(filepath):
        print(f'Skipping {name} — file not found: {filepath}')
        continue
    params = Params()
    params.action_type = 'fixed_policy'
    params.num_games = 20000
    params.state_mapping = 2
    params.fixed_policy_filepath = filepath
    game = BlackJack_game(params)
    game.play_games()
    num_wins = np.count_nonzero(game.sarsp_arr[:, 0] == game.win_state)
    results.append({'Policy': name, 'Win Rate': f'{num_wins / params.num_games:.1%}'})
    print()

pd.DataFrame(results)

Games played:   20000
Wins:           8531
Losses:         11469
Win percentage: 42.7%

Games played:   20000
Wins:           8442
Losses:         11558
Win percentage: 42.2%

Games played:   20000
Wins:           8455
Losses:         11545
Win percentage: 42.3%



,Policy,Win Rate
0,Q-Learning,42.7%
1,SARSA,42.2%
2,Value Iteration,42.3%
